# AGU manuscript figure exports

This notebook regenerates every result PDF requested by `agujournaltemplate.tex`. Figure 1 (the apparatus sketch) is an existing external drawing; Figures 2--6 are generated here from the ranked ORCA result CSV files and the digitized Ye & Ghassemi (2018) Table 2 data.

The output is vector PDF with embedded TrueType fonts. Loading and unloading remain ordered as stages 1--11: `8, 12, 16, 20, 24, 28, 24, 20, 16, 12, 8 MPa`. Hydraulic aperture is shown only as an informational quantity derived from flow and is not scored independently.

## 1. Paths and selected cases

Edit only this block when a final case or paper location changes. The path finder permits the notebook to be launched from the project root or any child directory.

In [1]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

def find_project_root(start=Path.cwd().resolve()):
    for candidate in (start, *start.parents):
        if (candidate / 'scripts' / 'export_agu_manuscript_figures.py').is_file():
            return candidate
    fallback = Path('/media/geomechanics/Data4TB/projects/orca_4.0')
    if (fallback / 'scripts' / 'export_agu_manuscript_figures.py').is_file():
        return fallback
    raise FileNotFoundError('Could not locate the ORCA project root')

PROJECT_ROOT = find_project_root()
PAPER_ROOT = Path.home() / 'Desktop' / 'AGU_Paper_1_PhD'
OUTPUT_DIR = PAPER_ROOT / 'Figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT / 'scripts') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))

import export_agu_manuscript_figures as paperfig

BB_CASES = {
    'SWT1': '107_01_swt1_coh27p2_apscale0p01512_ppfix',
    'SWT2': '100_04_swt2_apscale0p0177_ppfix',
    'SWS3': '100_06_sw3_resc1p30_unld0p00_ppfix',
    'SWS4': '93_07_sw4_final_theta30_jrc5_ppfix',
}

MC_CASES = {
    'SWT1': 'SWT1_OrcaMohrCoulombContactTraction_pb04',
    'SWT2': 'SWT2_OrcaMohrCoulombContactTraction_pb04',
    'SWS3': 'SWS3_OrcaMohrCoulombContactTraction_pb06',
    'SWS4': 'SWS4_OrcaMohrCoulombContactTraction_center',
}

MESH_CASES = dict(paperfig.MESH_CASES)
WEAKENING_CASES = dict(paperfig.WEAKENING_CASES)

print(f'Project: {PROJECT_ROOT}')
print(f'PDF destination: {OUTPUT_DIR}')

Project: /media/geomechanics/Data4TB/projects/orca_4.0
PDF destination: /home/geomechanics/Desktop/AGU_Paper_1_PhD/Figures


## 2. Preflight

This deliberately stops before plotting if a selected case is absent, incomplete in the ranking inventory, or has no result CSV. SWT1 files moved into `SWT1/Sweeps`; the resolver handles that reorganization without changing the historical ranking CSV.

In [2]:
selection_inventory = paperfig.preflight(BB_CASES, MC_CASES)
display(selection_inventory)
assert selection_inventory['stages'].eq('11/11').all(), 'A selected case is incomplete'

,model,sample,case,mean_nRMSE_pct,stages,result_csv
0,BBFast,SW-T1,107_01_swt1_coh27p2_apscale0p01512_ppfix,1.473366,11/11,Examples/YeGhasemmi2018/SWT1/Sweeps/results_cs...
1,BBFast,SW-T2,100_04_swt2_apscale0p0177_ppfix,2.131869,11/11,Examples/YeGhasemmi2018/SWT2/Sweeps/results_cs...
2,BBFast,SW-S3,100_06_sw3_resc1p30_unld0p00_ppfix,4.353781,11/11,Examples/YeGhasemmi2018/SWS3/Sweeps/results_cs...
3,BBFast,SW-S4,93_07_sw4_final_theta30_jrc5_ppfix,6.139187,11/11,Examples/YeGhasemmi2018/SWS4/Sweeps/results_cs...
4,Mohr-Coulomb,SW-T1,SWT1_OrcaMohrCoulombContactTraction_pb04,6.899642,11/11,Examples/YeGhasemmi2018/SWT1/results_csv_mc_sw...
5,Mohr-Coulomb,SW-T2,SWT2_OrcaMohrCoulombContactTraction_pb04,3.779733,11/11,Examples/YeGhasemmi2018/SWT2/results_csv_mc_sw...
6,Mohr-Coulomb,SW-S3,SWS3_OrcaMohrCoulombContactTraction_pb06,5.147762,11/11,Examples/YeGhasemmi2018/SWS3/results_csv_mc_sw...
7,Mohr-Coulomb,SW-S4,SWS4_OrcaMohrCoulombContactTraction_center,7.000445,11/11,Examples/YeGhasemmi2018/SWS4/results_csv_mc_sw...


In [3]:
exported = []

def save_pdf(figure_key, figure):
    path = OUTPUT_DIR / paperfig.FIGURE_FILENAMES[figure_key]
    figure.savefig(
        path,
        format='pdf',
        bbox_inches='tight',
        metadata={
            'Title': figure_key.replace('_', ' ').title(),
            'Author': 'ORCA Ye--Ghassemi validation workflow',
            'Subject': 'AGU manuscript result figure',
        },
    )
    exported.append(path)
    print(f'Saved {path.name} ({path.stat().st_size / 1024:.1f} kB)')
    return path

## 3. Figure 2 -- mesh sensitivity

The dumbbell plot compares the original 93-series BBFast baselines on production factor 5 and refined factor 3 meshes. The printed $\Delta$ is refined minus production mean nRMSE in percentage points.

In [4]:
fig2 = paperfig.figure_mesh_sensitivity(MESH_CASES)
save_pdf('mesh_sensitivity', fig2)
display(fig2)

Saved Figure_2_Mesh_Sensitivity.pdf (62.4 kB)


<Figure size 345x275 with 1 Axes>

## 4. Figure 3 -- four-specimen validation histories

Open symbols are the eleven Table 2 observations; blue curves are the selected BBFast stage states. The gray region is unloading. The figure contains exactly the five independent scored observables.

In [5]:
fig3 = paperfig.figure_validation_histories(BB_CASES)
save_pdf('validation_histories', fig3)
display(fig3)

Saved Figure_3_Validation_Histories.pdf (105.3 kB)


<Figure size 720x775 with 20 Axes>

## 5. Figure 4 -- coupled normal and hydraulic response

This shows normal displacement, hydraulic aperture, and validation-equivalent flow. The gray diamond aperture observations are explicitly labeled as derived from measured flow and are informational, not an extra validation channel.

In [6]:
fig4 = paperfig.figure_hydraulic_response(BB_CASES)
save_pdf('hydraulic_response', fig4)
display(fig4)

Saved Figure_4_Hydraulic_Response.pdf (112.6 kB)


<Figure size 720x750 with 12 Axes>

## 6. Figure 5 -- BBFast versus Mohr--Coulomb

Shear stress and displacement expose the loading-path separation. Stage 5 is highlighted because MC weakens one stage early for SW-T1, SW-T2, and SW-S3. The final panel reports the paired five-channel mean nRMSE values.

In [7]:
fig5 = paperfig.figure_bb_mc_comparison(BB_CASES, MC_CASES)
save_pdf('bb_mc_comparison', fig5)
display(fig5)

Saved Figure_5_BBFast_vs_MC.pdf (69.5 kB)


<Figure size 720x845 with 9 Axes>

## 7. Figure 6 -- weakening-exponent controls

The full shear-displacement histories compare the BBFast parent, exponent-1 BBFast control, matched 102-series MC transfer, and experiment. The yellow band identifies stage 5.

In [8]:
fig6 = paperfig.figure_weakening_controls(WEAKENING_CASES)
save_pdf('weakening_control', fig6)
display(fig6)

Saved Figure_6_Weakening_Controls.pdf (45.4 kB)


<Figure size 720x280 with 3 Axes>

## 8. Export manifest and LaTeX filenames

The manuscript can replace each placeholder with `\includegraphics[width=...]` using the paths printed below. Figure 2 is intended for `\linewidth`; Figures 3--6 are intended for `\textwidth`.

In [9]:
manifest = pd.DataFrame({
    'figure': range(2, 7),
    'filename': [paperfig.FIGURE_FILENAMES[key] for key in paperfig.FIGURE_FILENAMES],
    'exists': [(OUTPUT_DIR / paperfig.FIGURE_FILENAMES[key]).is_file() for key in paperfig.FIGURE_FILENAMES],
    'size_kB': [(OUTPUT_DIR / paperfig.FIGURE_FILENAMES[key]).stat().st_size / 1024 for key in paperfig.FIGURE_FILENAMES],
})
display(manifest)

for row in manifest.itertuples(index=False):
    width = r'\linewidth' if row.figure == 2 else r'\textwidth'
    print(rf'\includegraphics[width={width}]{{Figures/{row.filename}}}')

assert manifest['exists'].all(), 'At least one manuscript PDF was not exported'

,figure,filename,exists,size_kB
0,2,Figure_2_Mesh_Sensitivity.pdf,True,62.440430
1,3,Figure_3_Validation_Histories.pdf,True,105.254883
2,4,Figure_4_Hydraulic_Response.pdf,True,112.628906
3,5,Figure_5_BBFast_vs_MC.pdf,True,69.460938
4,6,Figure_6_Weakening_Controls.pdf,True,45.420898


\includegraphics[width=\linewidth]{Figures/Figure_2_Mesh_Sensitivity.pdf}
\includegraphics[width=\textwidth]{Figures/Figure_3_Validation_Histories.pdf}
\includegraphics[width=\textwidth]{Figures/Figure_4_Hydraulic_Response.pdf}
\includegraphics[width=\textwidth]{Figures/Figure_5_BBFast_vs_MC.pdf}
\includegraphics[width=\textwidth]{Figures/Figure_6_Weakening_Controls.pdf}


### One-cell rebuild

For later reruns, the following optional cell rebuilds and saves all five PDFs in one call. It duplicates the work above, so use it instead of Sections 3--7 when you do not need inline previews.

In [10]:
# all_figures, all_manifest = paperfig.export_all(
#     OUTPUT_DIR,
#     bb_cases=BB_CASES,
#     mc_cases=MC_CASES,
#     mesh_cases=MESH_CASES,
#     weakening_cases=WEAKENING_CASES,
#     close=False,
# )
# display(all_manifest)